In [1]:
import pandas as pd

path = "../data/raw/myntra202305041052.csv"
peek = pd.read_csv(path, nrows=5)
print(peek.columns.tolist())
peek.T


['id', 'name', 'img', 'asin', 'price', 'mrp', 'rating', 'ratingTotal', 'discount', 'seller', 'purl']


,0,1,2,3,4
id,1,2,3,4,5
name,Men Solid Oversized Cotton,Men Cotton Pure Cotton T-shirt,Women Pure Cotton T-shirt,Typography Print T-shirt,Printed Round Neck Pure Cotton T-shirt
img,"https://assets.myntassets.com/f_webp,dpr_1.0,q...","https://assets.myntassets.com/f_webp,dpr_1.0,q...","https://assets.myntassets.com/f_webp,dpr_1.0,q...","https://assets.myntassets.com/f_webp,dpr_1.0,q...","https://assets.myntassets.com/f_webp,dpr_1.0,q..."
asin,-,-,-,-,-
price,532,274,551,296,494
mrp,1299,499,1199,1099,899
rating,4.1,4.2,4.5,4.0,4.2
ratingTotal,5300,25400,3400,18400,3800
discount,59,45,54,73,45
seller,Difference of Opinion,Roadster,DILLINGER,Huetrap,Roadster


In [2]:
cols = ["id", "name", "price", "mrp", "rating", "ratingTotal", "discount", "seller", "purl"]

rows = 0
missing = pd.Series(0, index=cols)
ids = []
sample_parts = []
categories = pd.Series(dtype="float64")

for chunk in pd.read_csv(path, usecols=cols, chunksize=500_000, low_memory=False):
    rows += len(chunk)
    missing += chunk.isna().sum()
    ids.append(chunk["id"])
    sample_parts.append(chunk.sample(frac=0.002, random_state=42))
    path_part = chunk["purl"].str.split("/").str[3]
    categories = categories.add(path_part.value_counts(), fill_value=0)

ids = pd.concat(ids)
sample = pd.concat(sample_parts)

print("Rows:", rows)
print("Duplicate IDs:", ids.duplicated().sum())
print(missing.div(rows).sort_values(ascending=False))
print(categories.sort_values(ascending=False).head(30))


Rows: 1060213
Duplicate IDs: 0
id             0.0
name           0.0
price          0.0
mrp            0.0
rating         0.0
ratingTotal    0.0
discount       0.0
seller         0.0
purl           0.0
dtype: float64
purl
tshirts                  150918.0
kurta-sets                36405.0
sarees                    36285.0
bra                       31087.0
earrings                  27623.0
kurtas                    26671.0
bedsheets                 25371.0
jeans                     21796.0
watches                   21699.0
dresses                   20222.0
sunglasses                20058.0
jewellery-set             19542.0
trousers                  18845.0
tops                      18113.0
track-pants               18056.0
shorts                    18003.0
briefs                    16717.0
perfume-and-body-mist     16187.0
heels                     15876.0
night-suits               15277.0
lipstick                  14499.0
socks                     13091.0
flats                     1263

In [3]:
num = pd.read_csv(path, usecols=["id", "price", "mrp", "rating", "ratingTotal", "discount"])

print(num.dtypes)
print(num.describe().T)
print("rating == 0:", (num["rating"] == 0).sum())
print("ratingTotal == 0:", (num["ratingTotal"] == 0).sum())
print("price > mrp:", (num["price"] > num["mrp"]).sum())
print("mrp or price <= 0:", ((num["mrp"] <= 0) | (num["price"] <= 0)).sum())

calc = (1 - num["price"] / num["mrp"]) * 100
print("Discount differs by more than 1 point:", ((calc - num["discount"]).abs() > 1).sum())

id               int64
price          float64
mrp            float64
rating         float64
ratingTotal      int64
discount         int64
dtype: object
                 count           mean            std   min       25%  \
id           1060213.0  530107.000000  306057.274812   1.0  265054.0   
price        1060213.0    1536.235155    3051.140381  25.0     499.0   
mrp          1060213.0    2668.379334    3877.900441  25.0     999.0   
rating       1060213.0       1.060150       1.829632   0.0       0.0   
ratingTotal  1060213.0      41.896183     747.664535   0.0       0.0   
discount     1060213.0     149.641952     564.878137   0.0      15.0   

                  50%       75%        max  
id           530107.0  795160.0  1060213.0  
price           811.0    1497.0   257500.0  
mrp            1780.0    2999.0   257500.0  
rating            0.0       2.8        5.0  
ratingTotal       0.0       3.0    76400.0  
discount         50.0      68.0    19996.0  
rating == 0: 789750
ratingTo

In [4]:
mask = categories.index.str.contains("shirt|kurta|sneaker|shoe", na=False)
print(len(categories), "categories in total")
print(categories[mask].sort_values(ascending=False))

404 categories in total
purl
tshirts             150918.0
kurta-sets           36405.0
kurtas               26671.0
sweatshirts           8901.0
shirts                8392.0
sports-shoes          8319.0
casual-shoes          6762.0
lounge-tshirts        2288.0
shoe-accessories       184.0
formal-shoes            40.0
shoe-care               18.0
dtype: float64


In [5]:
calc = (1 - num["price"] / num["mrp"]) * 100
bad = num[(calc - num["discount"]).abs() > 1].copy()
bad["calc_discount"] = calc[bad.index].round(1)

print("discount > 100:", (num["discount"] > 100).sum())
print("calculated discount > 95:", (calc > 95).sum())
print("bad rows where discount equals mrp - price:", (bad["discount"] == bad["mrp"] - bad["price"]).sum())
bad.sample(10, random_state=42)


discount > 100: 84989
calculated discount > 95: 0
bad rows where discount equals mrp - price: 89791


,id,price,mrp,rating,ratingTotal,discount,calc_discount
1032953,1032954,1399.0,2796.0,0.0,0,1397,50.0
240603,240604,949.0,2199.0,0.0,0,1250,56.8
659368,659369,649.0,1497.0,0.0,0,848,56.6
909457,909458,349.0,999.0,0.0,0,650,65.1
291366,291367,419.0,3299.0,0.0,0,2880,87.3
315736,315737,440.0,945.0,3.8,5,505,53.4
490462,490463,999.0,2495.0,0.0,0,1496,60.0
785235,785236,1049.0,2399.0,0.0,0,1350,56.3
807912,807913,1199.0,3399.0,0.0,0,2200,64.7
632043,632044,820.0,4230.0,0.0,0,3410,80.6


In [6]:
want = ["shirts", "kurtas", "sports-shoes", "casual-shoes"]
parts = []

for chunk in pd.read_csv(path, usecols=["id", "name", "price", "mrp", "seller", "purl"], chunksize=500_000):
    chunk["cat_word"] = chunk["purl"].str.split("/").str[3]
    parts.append(chunk[chunk["cat_word"].isin(want)])

scope = pd.concat(parts, ignore_index=True)
print(len(scope), "rows in scope")

lower = scope["name"].str.lower()
scope["is_men"] = lower.str.contains(r"\bmen\b")
scope["is_women"] = lower.str.contains(r"\bwomen\b")
scope["is_kids"] = lower.str.contains(r"\b(?:boys?|girls?|kids?)\b")
scope["is_sneaker"] = lower.str.contains(r"\bsneakers?\b")

print(scope.groupby("cat_word").size())
print(scope.groupby("cat_word")[["is_men", "is_women", "is_kids", "is_sneaker"]].sum())

50144 rows in scope
cat_word
casual-shoes     6762
kurtas          26671
shirts           8392
sports-shoes     8319
dtype: int64
              is_men  is_women  is_kids  is_sneaker
cat_word                                           
casual-shoes       5      4672     1168        5407
kurtas          5054      6690      614           0
shirts           424       500     2621           0
sports-shoes     381      5751      618          14


In [7]:
p = scope["purl"].str.lower()
scope["url_men"] = p.str.contains(r"[/-]men-")
scope["url_women"] = p.str.contains(r"[/-]women-")
scope["url_kids"] = p.str.contains(r"[/-](?:boys?|girls?|kids?)-")

print(scope["purl"].sample(5, random_state=42).tolist())
print(scope.groupby("cat_word")[["url_men", "url_women", "url_kids"]].sum())

['https://www.myntra.com/kurtas/soch/soch-round-neck-beads-and-stones-straight-kurta/22239652/buy', 'https://www.myntra.com/shirts/marks--spencer/marks--spencer-women-blue--white-pure-cotton-striped-casual-shirt/17185316/buy', 'https://www.myntra.com/kurtas/rb/rb-round-neck-floral-printed-kurta/21251892/buy', 'https://www.myntra.com/shirts/us-polo-assn-kids/u-s-polo-assn-kids-boys-yellow-tartan-checks-opaque-checked-casual-shirt/22830124/buy', 'https://www.myntra.com/sports-shoes/adidas/adidas-women-floral-woven-design-pure-motion-20-running-shoes/20071832/buy']
              url_men  url_women  url_kids
cat_word                                  
casual-shoes        5       4925      1229
kurtas           6321       6547       676
shirts           1068       1839      4282
sports-shoes      426       6446       637
